In [7]:
import os
import re
import numpy as np
import pandas as pd

from mpire import WorkerPool


# =========================
# 路径设置
# =========================

DATA_DIR = "/share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea"

GENE_LIST_FILE = os.path.join(DATA_DIR, "gene_list.txt")

SNP_PATH = DATA_DIR

HAPDF_PATH = os.path.join(DATA_DIR, "2-allGeneHapDf")
HAPPER_PATH = os.path.join(DATA_DIR, "2-allGeneHapPer")

os.makedirs(HAPDF_PATH, exist_ok=True)
os.makedirs(HAPPER_PATH, exist_ok=True)

In [8]:
# =========================
# 参数设置
# =========================

GENE_COL = "GWAS signal"
CHR_COL = "Chr"
START_COL = "start"
END_COL = "end"

N_JOBS = 32

# 如果区间内没有 SNP，仍然输出文件，保证输出文件数量与 gene_list.txt 行数一致
# 这种情况下，所有样本都会被赋值为 Hap1
OUTPUT_EMPTY_REGION = False

# 是否像原始代码一样将样本名按 "_" 截断
# 原始代码中是 samples = [x.split("_")[0] for x in samples]
# 但 cowpea 的 snplist 样本名一般建议保留原样
SPLIT_SAMPLE_BY_UNDERSCORE = False


# =========================
# 工具函数
# =========================

def clean_columns(df):
    """
    清理列名中的 BOM 和首尾空格。
    """
    df.columns = [
        str(x).replace("\ufeff", "").strip()
        for x in df.columns
    ]
    return df


def normalize_chr(chr_value):
    """
    将染色体名称统一成 Chr01、Chr02 这种格式，
    用于匹配 Chr01.snplist、Chr02.snplist。
    """
    x = str(chr_value).strip()

    m = re.search(r"(\d+)", x)
    if m:
        return f"Chr{int(m.group(1)):02d}"

    if x.lower().startswith("chr"):
        return "Chr" + x[3:]

    return x


def clean_genotype_value(x):
    """
    清理 SNP 值。

    缺失值统一为 NA，避免 np.nan 在 tuple 中导致相同缺失模式不能正确合并。
    """
    if pd.isna(x):
        return "NA"

    if isinstance(x, str):
        x = x.strip()
        if x in ["", "-", ".", "./.", "NA", "NaN", "nan"]:
            return "NA"

    return x


def get_snp_file(chr_name):
    """
    根据染色体名称返回 snplist 文件路径。
    """
    return os.path.join(SNP_PATH, f"{chr_name}.snplist")


def find_chr_pos_cols(snp_df):
    """
    自动识别染色体列和位置列。

    你的文件正常应该是：
    第一列 #Chr
    第二列 Pos
    """
    cols = list(snp_df.columns)

    if "#Chr" in cols:
        chr_col = "#Chr"
    elif "Chr" in cols:
        chr_col = "Chr"
    else:
        chr_col = cols[0]

    if "Pos" in cols:
        pos_col = "Pos"
    elif "POS" in cols:
        pos_col = "POS"
    else:
        pos_col = cols[1]

    return chr_col, pos_col


def make_output_id(row):
    """
    输出文件名使用 Chr_start_end，而不是基因名。
    """
    chr_name = row["_chr_norm"]
    start = int(row[START_COL])
    end = int(row[END_COL])

    return f"{chr_name}_{start}_{end}"


def build_haplotype_for_region(snp_df, sample_cols, row):
    """
    针对 gene_list.txt 中的一行区间：
    1. 根据 start/end 提取 SNP
    2. 构建样本 × SNP 矩阵
    3. 构建 haplotype
    4. 统计 count 和 percentage
    5. 输出两个文件
    """
    output_id = row["_output_id"]
    chr_name = row["_chr_norm"]
    start = int(row[START_COL])
    end = int(row[END_COL])

    chr_col = row["_chr_col"]
    pos_col = row["_pos_col"]

    # 根据 start/end 筛选 SNP
    region_snp = snp_df[
        (snp_df[pos_col] >= start) &
        (snp_df[pos_col] <= end)
    ].copy()

    samples = sample_cols.copy()

    if SPLIT_SAMPLE_BY_UNDERSCORE:
        samples = [str(x).split("_")[0] for x in samples]

    # =========================
    # 情况1：区间内没有 SNP
    # 为了保证输出文件数量与 gene_list.txt 一致，仍然输出
    # 所有样本统一为 Hap1
    # =========================
    if region_snp.empty:
        if not OUTPUT_EMPTY_REGION:
            print(f"{output_id} SKIP: no SNP in {chr_name}:{start}-{end}")
            return

        geno_df = pd.DataFrame(index=samples)
        geno_df.index.name = "sample"
        geno_df["haplotype_id"] = "Hap1"

        hap_df = pd.DataFrame({
            "sample": samples,
            "haplotype_id": "Hap1",
            "count": len(samples),
            "percentage": 1.0
        })

        geno_df.to_csv(
            os.path.join(HAPDF_PATH, f"{output_id}.txt"),
            sep="\t"
        )

        hap_df[
            ["sample", "haplotype_id", "count", "percentage"]
        ].to_csv(
            os.path.join(HAPPER_PATH, f"{output_id}.txt"),
            sep="\t",
            index=False
        )

        print(
            f"{output_id} DONE: "
            f"{chr_name}:{start}-{end}, SNPs=0, Haplotypes=1"
        )

        return

    # 按 SNP 位置排序，保证单倍型组合顺序稳定
    region_snp = region_snp.sort_values(pos_col).copy()

    pos_list = region_snp[pos_col].astype(int).tolist()

    # 转成 样本 × SNP 矩阵
    geno_df = region_snp[sample_cols].T.copy()
    geno_df.index = samples
    geno_df.columns = pos_list
    geno_df.index.name = "sample"

    # 清理 SNP 值，缺失统一为 NA
    geno_for_hap = geno_df.applymap(clean_genotype_value)

    # 每个样本的所有 SNP 状态组合成一个 tuple，作为单倍型
    hap_strings = [
        tuple(row_values)
        for row_values in geno_for_hap.to_numpy()
    ]

    hap_df = pd.DataFrame({
        "sample": samples,
        "haplotype": hap_strings
    })

    # 按首次出现顺序编号 Hap1、Hap2、Hap3
    unique_haps = hap_df["haplotype"].unique()

    hap_to_id = {
        hap: f"Hap{idx + 1}"
        for idx, hap in enumerate(unique_haps)
    }

    hap_df["haplotype_id"] = hap_df["haplotype"].map(hap_to_id)

    # 统计每个 haplotype 的 count 和 percentage
    freq = (
        hap_df
        .groupby("haplotype_id")
        .size()
        .reset_index(name="count")
    )

    freq["percentage"] = freq["count"] / freq["count"].sum()

    freq2count = freq.set_index("haplotype_id")["count"].to_dict()
    freq2per = freq.set_index("haplotype_id")["percentage"].to_dict()

    hap_df["count"] = hap_df["haplotype_id"].map(freq2count)
    hap_df["percentage"] = hap_df["haplotype_id"].map(freq2per)

    sample2hap = (
        hap_df[["sample", "haplotype_id"]]
        .set_index("sample")
        .to_dict()["haplotype_id"]
    )

    geno_df["haplotype_id"] = geno_df.index.map(sample2hap)

    # 输出文件1：样本 × SNP 矩阵 + haplotype_id
    geno_df.to_csv(
        os.path.join(HAPDF_PATH, f"{output_id}.txt"),
        sep="\t"
    )

    # 输出文件2：每个样本对应 haplotype_id、count、percentage
    hap_df[
        ["sample", "haplotype_id", "count", "percentage"]
    ].to_csv(
        os.path.join(HAPPER_PATH, f"{output_id}.txt"),
        sep="\t",
        index=False
    )

    print(
        f"{output_id} DONE: "
        f"{chr_name}:{start}-{end}, "
        f"SNPs={len(pos_list)}, "
        f"Haplotypes={len(unique_haps)}"
    )


def process_one_chromosome(chr_name, rows):
    """
    处理一条染色体。

    注意：
    mpire.WorkerPool.map 会把 task = (chr_name, rows)
    自动拆成两个参数，所以这里必须写成：
    process_one_chromosome(chr_name, rows)
    """
    snp_file = get_snp_file(chr_name)

    if not os.path.exists(snp_file):
        raise FileNotFoundError(f"SNP file not found: {snp_file}")

    print(f"Reading {snp_file}")

    snp_df = pd.read_csv(
        snp_file,
        sep="\t",
        low_memory=False,
        na_values=["-", "./.", ".", "NA", "NaN", "nan"]
    )

    snp_df = clean_columns(snp_df)

    chr_col, pos_col = find_chr_pos_cols(snp_df)

    # Pos 转为数值
    snp_df[pos_col] = pd.to_numeric(snp_df[pos_col], errors="coerce")
    snp_df = snp_df.dropna(subset=[pos_col]).copy()
    snp_df[pos_col] = snp_df[pos_col].astype(int)

    # 按位置排序，提高后续筛选和输出稳定性
    snp_df = snp_df.sort_values(pos_col).copy()

    # 你的 snplist 文件中，前两列是 #Chr 和 Pos，后面都是品种 SNP
    sample_cols = list(snp_df.columns[2:])

    if len(sample_cols) == 0:
        raise ValueError(f"No sample columns found in {snp_file}")

    for row in rows:
        row["_chr_col"] = chr_col
        row["_pos_col"] = pos_col
        build_haplotype_for_region(snp_df, sample_cols, row)

    print(f"{chr_name} DONE")


def prepare_gene_info():
    """
    读取 gene_list.txt，
    检查必要列，
    标准化 Chr、start、end，
    生成输出文件名 Chr_start_end。
    """
    info_file = pd.read_csv(
        GENE_LIST_FILE,
        sep="\t",
        low_memory=False
    )

    info_file = clean_columns(info_file)

    required_cols = [
        GENE_COL,
        CHR_COL,
        START_COL,
        END_COL
    ]

    missing_cols = [
        col for col in required_cols
        if col not in info_file.columns
    ]

    if missing_cols:
        raise ValueError(
            "gene_list.txt 缺少必要列：{}\n当前列名为：{}".format(
                missing_cols,
                list(info_file.columns)
            )
        )

    # 删除必要信息缺失的行
    info_file = info_file.dropna(
        subset=[
            GENE_COL,
            CHR_COL,
            START_COL,
            END_COL
        ]
    ).copy()

    info_file[START_COL] = pd.to_numeric(
        info_file[START_COL],
        errors="coerce"
    )

    info_file[END_COL] = pd.to_numeric(
        info_file[END_COL],
        errors="coerce"
    )

    info_file = info_file.dropna(
        subset=[
            START_COL,
            END_COL
        ]
    ).copy()

    info_file[START_COL] = info_file[START_COL].astype(int)
    info_file[END_COL] = info_file[END_COL].astype(int)

    # 如果 start > end，自动交换
    swap_mask = info_file[START_COL] > info_file[END_COL]

    if swap_mask.any():
        info_file.loc[
            swap_mask,
            [START_COL, END_COL]
        ] = info_file.loc[
            swap_mask,
            [END_COL, START_COL]
        ].to_numpy()

    # 标准化染色体名，例如 Chr1、1、01 都变成 Chr01
    info_file["_chr_norm"] = info_file[CHR_COL].apply(normalize_chr)

    # 输出文件名前缀：Chr_start_end
    info_file["_output_id_base"] = info_file.apply(make_output_id, axis=1)

    # 为保证输出文件数量与 gene_list.txt 记录数一致，
    # 如果存在重复 Chr_start_end，则自动追加 row 编号，避免覆盖。
    duplicated_region = info_file["_output_id_base"].duplicated(keep=False)

    info_file["_output_id"] = info_file["_output_id_base"]

    info_file.loc[
        duplicated_region,
        "_output_id"
    ] = info_file.loc[
        duplicated_region
    ].apply(
        lambda x: f"{x['_output_id_base']}__row{x.name + 1}",
        axis=1
    )

    return info_file


# =========================
# 主程序
# =========================

if __name__ == "__main__":

    info_file = prepare_gene_info()

    print(f"Total regions: {info_file.shape[0]}")
    print(f"Total chromosomes: {info_file['_chr_norm'].nunique()}")

    # 按染色体分组。
    # 每条染色体只读取一次 ChrXX.snplist，避免重复读取大文件。
    tasks = []

    for chr_name, sub_df in info_file.groupby("_chr_norm", sort=True):
        rows = sub_df.to_dict("records")
        tasks.append((chr_name, rows))

    n_jobs = min(N_JOBS, len(tasks))

    with WorkerPool(n_jobs=n_jobs) as pool:
        pool.map(
            process_one_chromosome,
            tasks,
            progress_bar=True
        )

    print("ALL DONE")

Total regions: 91
Total chromosomes: 11


  0%|                                                                                                                                              | 0/11 [00:00<?, ?it/s]

Reading /share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/Chr01.snplistReading /share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/Chr02.snplistReading /share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/Chr04.snplistReading /share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/Chr03.snplist



Reading /share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/Chr07.snplistReading /share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/Chr06.snplistReading /share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/Chr05.snplist


Reading /share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/Chr10.snplistReading /share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/Chr11.snplist

Reading /share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/Chr08.snplist


  0%|                                                                                                                                              | 0/11 [00:00<?, ?it/s]

Reading /share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/Chr09.snplist


...

Chr03_56687574_56868067 DONE: Chr03:56687574-56868067, SNPs=223, Haplotypes=215


/tmp/ipykernel_271083/814845145.py:197: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  geno_for_hap = geno_df.applymap(clean_genotype_value)


Chr03_56866253_56866253 DONE: Chr03:56866253-56866253, SNPs=1, Haplotypes=4
Chr03_58091334_58091334 SKIP: no SNP in Chr03:58091334-58091334
Chr03_60821108_60821108 SKIP: no SNP in Chr03:60821108-60821108
Chr03_68974096_68974096 SKIP: no SNP in Chr03:68974096-68974096
Chr03 DONE


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [00:07<00:00,  3.08it/s]

ALL DONE
